# SMS Spam Classifier — Transformer from scratch

A small Transformer encoder built and trained from scratch in PyTorch on the [SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection) dataset (5,572 messages). Reaches ~97% test accuracy.

In [ ]:
import pandas as pd
import torch
import torch.nn as nn

## Load the dataset

In [2]:
df = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "text"]
)

print(df.head())
print(df.shape)
print(df["label"].value_counts())

  label                                               text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
(5572, 2)
label
ham     4825
spam     747
Name: count, dtype: int64


## Split into train / validation / test and encode labels

In [3]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (3900, 2)
Validation: (836, 2)
Test: (836, 2)


In [4]:
train_df["label"] = train_df["label"].map({
    "ham": 0,
    "spam": 1
})

test_df["label"] = test_df["label"].map({
    "ham": 0,
    "spam": 1
})

val_df["label"] = val_df["label"].map({
    "ham": 0,
    "spam": 1
})

## Build the vocabulary

Words appearing at least twice in the training set get an ID; everything else maps to `<UNK>`. The vocabulary is built from the training split only, so no information leaks from validation/test.

In [ ]:
from collections import Counter

word_counts = Counter()

for text in train_df["text"]:
    tokens = text.lower().split()
    word_counts.update(tokens)

vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word, count in word_counts.items():
    if count >= 2:
        vocab[word] = len(vocab)

## Dataset and DataLoaders

Each SMS becomes a fixed-length sequence of 64 token IDs (truncated or padded with `<PAD>`).

In [6]:
from torch.utils.data import Dataset

class SMSSpamCollection(Dataset):
    def __init__(self, data, vocab, max_length=64):
        self.data = data
        self.vocab = vocab
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data.loc[idx, "text"]
        label = self.data.loc[idx, "label"]

        words = text.lower().split() # 1. split text into words and convert to lowercase

        token_ids = [
            self.vocab.get(word, self.vocab["<UNK>"])
            for word in words        # 2. convert words to token IDs from vocab function
        ]

        token_ids = token_ids[:self.max_length]

        padding = self.max_length - len(token_ids)  # 3. truncate or pad the token IDs to max_length
                                                    # so that all sequences have the same length    
        token_ids += [self.vocab["<PAD>"]] * padding

        return (
            torch.tensor(token_ids, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

In [7]:
from torch.utils.data import DataLoader
# dataloader makes a batch of data and shuffles the data for training
train_dataset = SMSSpamCollection(train_df.reset_index(drop=True), vocab)
test_dataset = SMSSpamCollection(test_df.reset_index(drop=True), vocab)
val_dataset = SMSSpamCollection(val_df.reset_index(drop=True), vocab)
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=16, shuffle=True)

## Model

Word embeddings + learned positional embeddings → 2-layer Transformer encoder → masked mean pooling → linear classifier (ham/spam).

In [8]:
# positional encoding creates a vector for each position
# x = word_vectors + position_vectors 

class PositionalEncoding(nn.Module):    
    def __init__(self):
        super().__init__()
        self.position_embedding = nn.Embedding(64, 64)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device)
        return x + self.position_embedding(positions)

In [12]:
class TransformerModel(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()

        self.word_embedding = nn.Embedding(vocab_size, 64, padding_idx=0)
        self.position_encoding = PositionalEncoding()

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=64,
            nhead=4,
            dim_feedforward=128,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2,
            enable_nested_tensor=False  # the nested-tensor fast path crashes
                                        # with a CUDA error on this GPU
        )

        self.classifier = nn.Linear(64, 2)
        
    def forward(self, input_ids):
        x = self.word_embedding(input_ids)
        x = self.position_encoding(x)

        padding_mask = input_ids == 0   # True where the token is <PAD>,
                                        # so attention ignores those positions
        x = self.transformer(x, src_key_padding_mask=padding_mask)

        mask = input_ids != 0       # Pooling combines the contextual word vectors into one 1
        mask = mask.unsqueeze(-1)   # vector for the whole SMS.2

        x = x * mask                # 3
        x = x.sum(dim=1) / mask.sum(dim=1).clamp(min=1) # 4

        return self.classifier(x)

In [13]:
model = TransformerModel(
    vocab_size=len(vocab),
)

inputs, labels = next(iter(train_dataloader))

output = model(inputs)

print(output.shape)

torch.Size([16, 2])


## Training

Train for 10 epochs with AdamW, tracking validation loss each epoch and keeping the weights from the best epoch.

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [17]:
model = TransformerModel(
    vocab_size=len(vocab)
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001
)

loss_function = nn.CrossEntropyLoss()

In [18]:
best_val_loss = float("inf")
best_model_state = None

for epoch in range(10):
    model.train()
    train_loss = 0

    for inputs, labels in train_dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = loss_function(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = loss_function(outputs, labels)

            val_loss += loss.item()

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    train_loss /= len(train_dataloader)
    val_loss /= len(val_dataloader)
    val_accuracy = correct / total

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = {
            name: parameter.detach().cpu().clone()
            for name, parameter in model.state_dict().items()
        }

    print(
        f"Epoch {epoch + 1}: "
        f"train loss={train_loss:.4f}, "
        f"val loss={val_loss:.4f}, "
        f"val accuracy={val_accuracy:.4f}"
    )

Epoch 1: train loss=0.2112, val loss=0.1084, val accuracy=0.9581
Epoch 2: train loss=0.0793, val loss=0.0698, val accuracy=0.9785
Epoch 3: train loss=0.0423, val loss=0.0680, val accuracy=0.9809
Epoch 4: train loss=0.0242, val loss=0.0709, val accuracy=0.9773
Epoch 5: train loss=0.0158, val loss=0.0726, val accuracy=0.9773
Epoch 6: train loss=0.0114, val loss=0.1247, val accuracy=0.9665
Epoch 7: train loss=0.0235, val loss=0.0696, val accuracy=0.9833
Epoch 8: train loss=0.0095, val loss=0.0894, val accuracy=0.9761
Epoch 9: train loss=0.0155, val loss=0.0866, val accuracy=0.9761
Epoch 10: train loss=0.0107, val loss=0.0889, val accuracy=0.9761


## Evaluate on the test set

Restore the best-epoch weights, then measure accuracy, confusion matrix, and per-class precision/recall.

In [19]:
# load the weights from the epoch with the lowest validation loss,
# instead of testing whatever the model looked like after the last epoch
model.load_state_dict(best_model_state)
model.eval()

correct = 0
total = 0
test_loss = 0

with torch.no_grad():
    for inputs, labels in test_dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        loss = loss_function(outputs, labels)

        test_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

print("Test loss:", test_loss / len(test_dataloader))
print("Test accuracy:", correct / total)

Test loss: 0.12066436895248392
Test accuracy: 0.9688995215311005


In [21]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_dataloader:
        inputs = inputs.to(device)

        outputs = model(inputs)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(labels.tolist())

print(confusion_matrix(all_labels, all_predictions))

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["ham", "spam"],
        digits=4
    )
)

[[713  11]
 [ 15  97]]
              precision    recall  f1-score   support

         ham     0.9794    0.9848    0.9821       724
        spam     0.8981    0.8661    0.8818       112

    accuracy                         0.9689       836
   macro avg     0.9388    0.9254    0.9320       836
weighted avg     0.9685    0.9689    0.9687       836



## Save the model checkpoint

The checkpoint bundles the weights with the vocabulary and config, so inference code can rebuild everything from one file. This is the file served by the FastAPI app (`app.py` / `inference.py`).

In [22]:
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "vocab": vocab,
        "max_length": 64,
        "model_config": {
            "embedding_size": 64,
            "num_heads": 4,
            "num_layers": 2,
            "feedforward_size": 128
        },
        "label_to_id": {
            "ham": 0,
            "spam": 1
        }
    },
    "spam_transformer.pth"
)

In [23]:
import json

with open("vocab.json", "w", encoding="utf-8") as file:
    json.dump(vocab, file)